# Projet Home Credit — Sujet 03

*Séance :* 01  
*Nom :* HOUNYO Charbel 

*Porteur :* P2

**Étapes assignées :**

- 1.2 : Charger et compter
- 1.3 : Type
- 1.5 : Valeur Impossible
- 1.10 : Plancher


# Carnet de démarrage — sujet 03 · Home Credit

**Objectif du sujet** : décider d'accorder ou de refuser un crédit.

Ce carnet est un **châssis**, pas une solution. Il fait quatre choses et s'arrête :

1. il **trouve** vos fichiers, quelle que soit la plateforme ;
2. il **charge** la table efficacement — cache Parquet construit une fois, projection de colonnes ;
3. il **profile** les données : dimensions, types, cible, manquants ;
4. il **écrit** une soumission au bon format, avec une valeur de référence volontairement nulle.

**Ce qu'il ne fait pas** : choisir votre métrique, votre découpage de validation, votre modèle, ni vos
variables. Ces décisions sont le sujet, et c'est ce que la grille note.

8,07 % de défauts. Les manquants ne sont pas comblés : c'est une information.

---

## Où mettre ce carnet

| Plateforme | Ce qu'il faut faire |
|---|---|
| **Le dossier du sujet** | **https://drive.google.com/drive/folders/17v1qEMO8FohlDPU99fAjEcuy_nl-Nim5?usp=sharing** — téléchargez-le avant tout (le lien figure aussi dans l'énoncé) |
| **En local (VS Code, Jupyter)** | placez le carnet **dans le dossier téléchargé**, ou définissez `DONNEES_PROJET` vers ce dossier |
| **Google Colab** | déposez les fichiers dans `MyDrive/Sujet_03_Home_Credit/` — le carnet les trouvera ; ou téléversez-les dans `/content/` |
| **Kaggle** | créez un jeu de données avec vos fichiers, **Add Input** — `/kaggle/input` est exploré automatiquement |

**Rien n'est codé en dur** : le carnet cherche, et s'il ne trouve pas, il **dit précisément** ce qu'il a
cherché et où.


## Les fichiers de ce sujet

| Fichier | Rôle |
|---|---|
| `train.csv` | entraînement ou annexe |
| `a_decider.csv` | à prédire — la cible est absente |
| `HomeCredit_columns_description.csv` | entraînement ou annexe |

**Format de sortie attendu** : `SK_ID_CURR,TARGET`.
**Colonne d'identifiant** : `SK_ID_CURR`.
**Fichier à produire** : `soumission.csv`, à la racine de votre dépôt.


## Sur les données lourdes — et pourquoi pas « raylib »

`raylib` est une bibliothèque **graphique** (fenêtres, dessin, jeux) : elle ne lit aucune table. Elle
n'a rien à faire ici. Les outils qui répondent au besoin sont :

| Outil | Rôle | Présent par défaut ? |
|---|---|---|
| **Parquet + pyarrow** | format colonne compressé, lecture sélective | **oui** avec pandas récent |
| **Polars** | moteur colonne, `scan_csv` paresseux | non — `pip install polars` |
| **DuckDB** | SQL directement sur CSV/Parquet, sans chargement | non — `pip install duckdb` |

Ce carnet utilise **Parquet** (garanti), et le remplace par Polars ou DuckDB si vous les installez. Le
principe ne change pas : **ne jamais charger ce dont on n'a pas besoin**, et **payer la conversion
CSV → Parquet une seule fois**.


In [1]:
# --- 1. Réglages --------------------------------------------------------------
from pathlib import Path
import os
import pandas as pd

# Où télécharger vos données — le dossier de votre sujet sur le Drive du cours.
LIEN_DRIVE = "__LIEN__"

# ECHANTILLON : nombre de lignes lues pour démarrer. None = tout lire.
ECHANTILLON = 200_000
DOSSIER_CACHE = Path.cwd() / "_cache_parquet"


In [2]:
# --- 2. Trouver les données, sans rien coder en dur --------------------------
FICHIERS_ATTENDUS = ['train.csv', 'a_decider.csv', 'HomeCredit_columns_description.csv']
SUJET = "Sujet_03_Home_Credit"


def trouver_donnees(fichiers=FICHIERS_ATTENDUS, sujet=SUJET):
    """Cherche le dossier qui contient TOUS les fichiers attendus."""
    essais = []
    pistes = []
    if os.environ.get("DONNEES_PROJET"):
        pistes.append(Path(os.environ["DONNEES_PROJET"]))
    ici = Path.cwd().resolve()
    pistes += [ici, ici / "donnees", ici / sujet]
    pistes += list(ici.parents)[:4]
    pistes += [Path("/kaggle/input"), Path("/content"), Path("/content/drive/MyDrive")]

    for racine in pistes:
        if not racine.exists():
            continue
        candidates = [racine, racine / sujet]
        if racine.name in ("input", "MyDrive"):
            candidates += [d for d in racine.iterdir() if d.is_dir()]
        for c in candidates:
            essais.append(str(c))
            if all((c / f).is_file() for f in fichiers):
                return c
    raise FileNotFoundError(
        "Dossier de données introuvable.\n"
        "Il faut un dossier contenant : " + ", ".join(fichiers) + "\n"
        "Solutions : placez le carnet à côté de ce dossier, "
        "ou définissez DONNEES_PROJET=/chemin/vers/le/dossier.\n"
        "Chemins essayés : " + str(essais[:12])
    )


DOSSIER = trouver_donnees()
print("dossier de données :", DOSSIER)
print("fichiers vus :", sorted(p.name for p in DOSSIER.iterdir() if p.is_file())[:12])


dossier de données : C:\Users\HP\Downloads\HomeCredit
fichiers vus : ['Carnet_de_demarrage.ipynb', 'Enonce.md', 'HomeCredit_columns_description.csv', 'a_decider.csv', 'soumission.csv', 'train.csv']


In [3]:
# --- 3. Chargement efficace (table lourde) ------------------------------------
# Le cache Parquet s'écrit à côté du carnet, jamais dans les données : sur Kaggle,
# /kaggle/input est en lecture seule.
import pyarrow as pa
import pyarrow.csv as pvc
import pyarrow.parquet as pq


def table(nom, colonnes=None, complet=False):
    """Ouvre un CSV en profitant d'un cache Parquet.

    nom      : nom du fichier dans le dossier de données
    colonnes : liste de colonnes à garder (None = toutes) — le levier principal
    morceaux : si un entier, rend un itérateur de DataFrames de cette taille
    """
    source = DOSSIER / nom
    if not source.is_file():
        raise FileNotFoundError(f"{nom} introuvable dans {DOSSIER}")

    # lecture d'un échantillon : pas de cache, on veut seulement voir
    if ECHANTILLON and not complet:
        lots, reste = [], ECHANTILLON
        for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 25)):
            lots.append(lot)
            reste -= lot.num_rows
            if reste <= 0:
                break
        t = pa.Table.from_batches(lots)
        if colonnes:
            t = t.select([c for c in colonnes if c in t.column_names])
        return t.to_pandas()

    # cache Parquet, construit une fois
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    if not cache.is_file():
        DOSSIER_CACHE.mkdir(parents=True, exist_ok=True)
        print(f"construction du cache Parquet : {cache.name} (une seule fois)")
        ecrivain = None
        try:
            for lot in pvc.open_csv(str(source), read_options=pvc.ReadOptions(block_size=1 << 26)):
                if colonnes:
                    lot = lot.select([c for c in colonnes if c in lot.schema.names])
                if ecrivain is None:
                    ecrivain = pq.ParquetWriter(cache, lot.schema)
                ecrivain.write_batch(lot)
        finally:
            if ecrivain is not None:
                ecrivain.close()
    fichier = pq.ParquetFile(cache)
    return fichier.read(columns=colonnes).to_pandas()


def table_par_morceaux(nom, colonnes=None, morceaux=200_000):
    """Itère sur la table par paquets — pour les traitements qui ne tiennent pas en mémoire."""
    table(nom, colonnes=colonnes)          # garantit que le cache existe
    cache = DOSSIER_CACHE / (Path(nom).stem + ".parquet")
    for lot in pq.ParquetFile(cache).iter_batches(batch_size=morceaux):
        yield lot.to_pandas()


In [4]:
# --- 4. Charger et regarder ---------------------------------------------------
entrainement = table("train.csv")
print("dimensions :", entrainement.shape)
print("colonnes :", len(entrainement.columns))
entrainement.head()


dimensions : (246008, 122)
colonnes : 122


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,123551,0,Cash loans,M,N,Y,0,202500.0,1078200.0,31653.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,400429,0,Cash loans,F,N,Y,2,148500.0,544491.0,20133.0,...,0,0,0,0,0.0,0.0,0.0,0.0,2.0,0.0
2,106912,0,Cash loans,F,Y,N,2,67500.0,363190.5,24268.5,...,0,0,0,0,0.0,0.0,0.0,1.0,1.0,3.0
3,383991,0,Revolving loans,F,N,Y,1,144000.0,450000.0,22500.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,4.0
4,423763,0,Cash loans,F,N,Y,0,67500.0,675000.0,21775.5,...,0,0,0,0,0.0,0.0,1.0,0.0,0.0,0.0


## Étape 1 — charger et compter

Avant de créer un modèle, vérifie les dimensions exactes de chaque fichier et compte les dossiers de chaque catégorie de `TARGET` dans `train.csv` (`0` = pas de défaut connu, `1` = défaut). Le total des catégories doit égaler le nombre de lignes d'entraînement. Le pourcentage de `TARGET = 1` permet de voir si les catégories sont déséquilibrées, ce qui compte pour choisir ensuite comment évaluer le modèle. Cette cellule lit les fichiers complets, pas seulement l'aperçu affiché plus haut.

In [5]:
from collections import Counter

for nom in FICHIERS_ATTENDUS:
    lecteur = pvc.open_csv(str(DOSSIER / nom))
    nombre_lignes = 0
    repartition_cible = Counter()

    for lot in lecteur:
        nombre_lignes += lot.num_rows
        if nom == "train.csv":
            repartition_cible.update(lot.column("TARGET").to_pylist())

    print(f"{nom} : {nombre_lignes:,} lignes et {len(lecteur.schema.names)} colonnes")

    if nom == "train.csv":
        assert sum(repartition_cible.values()) == nombre_lignes
        print("Répartition de TARGET :")
        for valeur, nombre in sorted(repartition_cible.items()):
            pourcentage = 100 * nombre / nombre_lignes
            print(f"  {valeur} : {nombre:,} ({pourcentage:.2f} %)")

train.csv : 246,008 lignes et 122 colonnes
Répartition de TARGET :
  0 : 226,148 (91.93 %)
  1 : 19,860 (8.07 %)
a_decider.csv : 61,503 lignes et 121 colonnes
HomeCredit_columns_description.csv : 219 lignes et 5 colonnes


## Étape 2 — identifier le type des colonnes

Le type indique à Python comment une valeur est représentée : entier (`int`), nombre décimal (`float`) ou texte (`object`/`string`). C'est utile pour repérer les colonnes numériques et catégorielles avant le traitement. Ici, on inspecte l'échantillon d'entraînement déjà chargé (`entrainement`) ; le carnet en charge jusqu'à 200 000 lignes pour l'exploration.

In [6]:
types_colonnes = entrainement.dtypes.astype(str).rename("type").to_frame()
types_colonnes["valeurs distinctes dans l'échantillon"] = entrainement.nunique(dropna=True)

print("Nombre de colonnes par type :")
display(types_colonnes["type"].value_counts().rename_axis("type").to_frame("nombre de colonnes"))

print("Type et nombre de valeurs distinctes de chaque colonne :")
display(types_colonnes)

Nombre de colonnes par type :


,nombre de colonnes
type,
float64,65
int64,41
str,16


Type et nombre de valeurs distinctes de chaque colonne :


,type,valeurs distinctes dans l'échantillon
SK_ID_CURR,int64,246008
TARGET,int64,2
NAME_CONTRACT_TYPE,str,2
CODE_GENDER,str,3
FLAG_OWN_CAR,str,2
...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,float64,9
AMT_REQ_CREDIT_BUREAU_WEEK,float64,9
AMT_REQ_CREDIT_BUREAU_MON,float64,23
AMT_REQ_CREDIT_BUREAU_QRT,float64,11


## Étape 3 — repérer les valeurs impossibles

On vérifie quelques règles de cohérence simples sur **tout** `train.csv` : cible binaire, identifiants positifs et uniques, montants et compteurs non négatifs, indicateurs documentaires binaires et colonnes de jours cohérentes. `DAYS_EMPLOYED = 365243` est un code spécial connu, donc il n'est pas compté comme une erreur. La cellule signale les anomalies et quelques exemples, mais ne corrige rien automatiquement : une valeur inhabituelle doit être comprise avant d'être modifiée.

In [7]:
source_train = DOSSIER / "train.csv"
colonnes_train = pvc.open_csv(str(source_train)).schema.names

regles = {
    "TARGET": lambda s: ~s.isin([0, 1]),
    "SK_ID_CURR": lambda s: s <= 0,
    "DAYS_BIRTH": lambda s: s > 0,
    "DAYS_REGISTRATION": lambda s: s > 0,
    "DAYS_ID_PUBLISH": lambda s: s > 0,
    "DAYS_LAST_PHONE_CHANGE": lambda s: s > 0,
    "DAYS_EMPLOYED": lambda s: (s > 0) & (s != 365243),
}

for colonne in colonnes_train:
    if colonne.startswith(("AMT_", "CNT_", "OBS_", "DEF_")):
        regles[colonne] = lambda s: s < 0
    elif colonne.startswith("FLAG_DOCUMENT_"):
        regles[colonne] = lambda s: ~s.isin([0, 1])

regles = {colonne: regle for colonne, regle in regles.items() if colonne in colonnes_train}
colonnes_a_lire = list(regles)
if "SK_ID_CURR" not in colonnes_a_lire:
    raise ValueError("La colonne SK_ID_CURR est absente de train.csv")

anomalies = {colonne: {"nombre": 0, "exemples": []} for colonne in colonnes_a_lire}
identifiants_vus = set()
doublons_identifiant = 0

lecteur_verification = pvc.open_csv(
    str(source_train),
    convert_options=pvc.ConvertOptions(include_columns=colonnes_a_lire),
)
for lot_verification in lecteur_verification:
    bloc = lot_verification.to_pandas()
    for identifiant in bloc["SK_ID_CURR"].dropna():
        if identifiant in identifiants_vus:
            doublons_identifiant += 1
        identifiants_vus.add(identifiant)

    for colonne, regle in regles.items():
        valeurs = bloc[colonne].dropna()
        invalides = valeurs[regle(valeurs).fillna(False)]
        anomalies[colonne]["nombre"] += len(invalides)
        exemples = anomalies[colonne]["exemples"]
        exemples.extend(invalides.head(5 - len(exemples)).tolist())

print("Anomalies selon les règles de cohérence :")
anomalies_trouvees = False
for colonne, resultat in anomalies.items():
    if resultat["nombre"]:
        anomalies_trouvees = True
        print(f"{colonne} : {resultat['nombre']:,} valeur(s) à vérifier ; exemples : {resultat['exemples']}")
if not anomalies_trouvees:
    print("Aucune valeur ne viole les règles vérifiées.")
print(f"Identifiants SK_ID_CURR dupliqués : {doublons_identifiant:,}")

Anomalies selon les règles de cohérence :
Aucune valeur ne viole les règles vérifiées.
Identifiants SK_ID_CURR dupliqués : 0


## Étape 4 — mesurer un plancher

`TARGET = 1` signifie qu'un dossier se termine en défaut. Avant d'entraîner un modèle, on mesure deux règles extrêmes qui n'utilisent aucune variable :

- **Accorder toutes les demandes** (`TARGET = 0`) : c'est notre plancher principal, une règle simple qui ne sait pas repérer les défauts. Elle permet notamment de voir pourquoi l'exactitude peut être trompeuse avec seulement 8,07 % de défauts.
- **Refuser toutes les demandes** (`TARGET = 1`) : ce repère complémentaire montre l'autre extrême. Il obtient un rappel parfait, mais sa précision n'est que la proportion de défauts ; un rappel élevé ne suffit donc pas à prouver qu'une règle est utile.

On compare précision, rappel et exactitude sur la même validation, créée avec un découpage stratifié reproductible (80 % entraînement, 20 % validation). La précision et le rappel sont calculés en considérant le défaut (`TARGET = 1`) comme la classe positive. Pour la règle « tout accorder », la précision vaut 0 par convention (`zero_division=0`), puisqu'aucun dossier n'est prédit en défaut.

La métrique finale et les coûts des deux erreurs restent à décider avec l'équipe. Les futurs modèles devront être évalués sur **cette même validation** ; les transformations et imputations devront ensuite être ajustées sur l'entraînement seulement.

In [8]:
# --- 4. Mesurer le plancher ---------------------------------------------------
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Charger toutes les lignes ; ECHANTILLON ne s'applique pas avec complet=True.
train_complet = table("train.csv", complet=True)
X = train_complet.drop(columns=["TARGET", "SK_ID_CURR"])
y = train_complet["TARGET"].astype("int8")

# Ce découpage sera conservé pour comparer équitablement les modèles ensuite.
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

planchers = {
    "Tout accorder (plancher principal)": np.zeros(len(y_validation), dtype="int8"),
    "Tout refuser (repère complémentaire)": np.ones(len(y_validation), dtype="int8"),
}

print(f"Taux de défaut dans la validation : {y_validation.mean():.2%}")
for nom, predictions in planchers.items():
    precision = precision_score(y_validation, predictions, zero_division=0)
    rappel = recall_score(y_validation, predictions, zero_division=0)
    exactitude = accuracy_score(y_validation, predictions)
    print(f"{nom} : précision={precision:.2%} · rappel={rappel:.2%} · exactitude={exactitude:.2%}")

construction du cache Parquet : train.parquet (une seule fois)
Taux de défaut dans la validation : 8.07%
Tout accorder (plancher principal) : précision=0.00% · rappel=0.00% · exactitude=91.93%
Tout refuser (repère complémentaire) : précision=8.07% · rappel=100.00% · exactitude=8.07%


In [9]:
# --- 5. Profil de la cible ----------------------------------------------------
CIBLE = "TARGET"
if CIBLE in entrainement.columns:
    serie = entrainement[CIBLE]
    print("type :", serie.dtype)
    print("manquants :", int(serie.isna().sum()))
    print(serie.describe())
    if serie.nunique() <= 20:
        print(serie.value_counts(normalize=True).sort_index())
else:
    print(f"la cible {CIBLE} est absente de cette table — normal pour la table à prédire")


type : int64
manquants : 0
count    246008.000000
mean          0.080729
std           0.272419
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: TARGET, dtype: float64
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


In [10]:
# --- 6. Soumission de RÉFÉRENCE (format seulement) ----------------------------
# Cette valeur n'a AUCUNE prétention prédictive : elle vérifie le format.
COLONNES_SORTIE = ['SK_ID_CURR', 'TARGET']


def ecrire_soumission(identifiants, valeurs=None, chemin="soumission.csv"):
    """Écrit le fichier attendu. `valeurs` peut être une colonne, ou une matrice de probabilités."""
    sortie = pd.DataFrame({"SK_ID_CURR": identifiants})
    if len(COLONNES_SORTIE) == 2:
        sortie["TARGET"] = valeurs
    else:
        n = len(COLONNES_SORTIE) - 1
        matrice = valeurs if valeurs is not None else [[1.0 / n] * n for _ in identifiants]
        for j, colonne in enumerate(COLONNES_SORTIE[1:]):
            sortie[colonne] = [ligne[j] for ligne in matrice]
    sortie.to_csv(chemin, index=False)
    print(f"écrit : {chemin} · {len(sortie):,} lignes · colonnes {list(sortie.columns)}")
    if len(COLONNES_SORTIE) > 2:
        print("somme par ligne (doit valoir 1) :", sortie.iloc[:, 1:].sum(axis=1).round(6).unique()[:3])
    return sortie


a_predire = table("a_decider.csv", colonnes=["SK_ID_CURR"], complet=True)
identifiants = a_predire["SK_ID_CURR"].drop_duplicates().to_numpy()
print("identifiants à prédire :", len(identifiants))
ecrire_soumission(identifiants)


identifiants à prédire : 61503
écrit : soumission.csv · 61,503 lignes · colonnes ['SK_ID_CURR', 'TARGET']


,SK_ID_CURR,TARGET
0,424613,None
1,260084,None
2,214262,None
3,357905,None
4,101573,None
...,...,...
61498,342357,None
61499,382938,None
61500,235122,None
61501,348296,None


In [11]:
# --- 7. Le travail commence ICI -----------------------------------------------
# À vous de décider, et de justifier :
#   - la métrique, et ce qu'elle coûte ;
#   - le découpage de validation ;
#   - les variables, leur encodage, leur imputation ;
#   - le modèle, son réglage, son seuil.
# Ce carnet ne le fera pas à votre place : c'est l'objet du projet.
raise NotImplementedError(
    "Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici."
)


NotImplementedError: Carnet de démarrage : la modélisation est à votre charge. Écrivez votre pipeline à partir d'ici.